# Iceland2 river forcing

This notebook builds the river forcing file for Iceland2. It holds the GloFAS rivers across the domain and the Hvalfjörður rivers from the custom dataset `Hvalfjordur_rivers_2024.nc`. Run it on Anvil, where the GloFAS data lives. It writes the file that `Iceland2_glodap_woa.forge_blueprint.yaml` names as the river `custom_file`, and prints the content hash to record there.

| | GloFAS rivers | Hvalfjörður rivers |
|---|---|---|
| Discharge | GloFAS v4, daily; rivers with a mean under 1 m³/s set to 0 (roms-tools default) | Custom dataset: every month is half the August flow, and 40 % of that in January–April and November–December; 2023 repeats 2024 |
| Temperature | 0–10 °C over the year | 0–10 °C over the year |
| Salinity | 1 psu (roms-tools default) | 1 psu |
| BGC tracers | roms-tools defaults (`river_tracer_defaults.nc`, recommended values) | Table S1: ALK 282, DIC 313, NO₃ 12, PO₄ 0.9, SiO₃ 14, NH₄ 0.7, O₂ 300 mmol m⁻³; every other tracer 0, Fe and Lig included |

The temperature is the sinusoid of the earlier version, 10 sin(π (m − 1)/11) °C on the 15th of month m, interpolated linearly in between as ROMS interpolates monthly records. GloFAS rivers that enter the fjord are dropped, because the custom dataset covers it.

**Changes from the earlier version.** The earlier version of this notebook wrote `Iceland2_river_combied.nc`, the file the `iceland2_customriver` runs used. That file has problems:
- it holds the Hvalfjörður rivers only: the last cell saved `riv`, not the GloFAS + Hvalfjörður combination;
- the combination itself numbered the Hvalfjörður rivers 1–16 on top of GloFAS rivers 1–16, and it kept the discharge of the GloFAS rivers it removed from the fjord;
- the notebook last ran before the Table S1 values went in, so the file most likely has PO₄ 0.4, NO₃ 6, SiO₃ 3, NH₄ 0.4 and O₂ 360 (checked at the end of this notebook);
- `xr.concat` gave `river_index` and `river_fraction` a time dimension, and `to_netcdf` stored the times as int64, which ParallelIO cannot read.

Here, rivers are numbered once (GloFAS first) on GloFAS's daily time axis. The file is written as CDF5 with every value in double precision.

In [ ]:
import hashlib
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from roms_tools import Grid, RiverForcing

## Settings

In [ ]:
# The model grid forge wrote for the custom-river run. Iceland2_glodap_woa uses the same
# file (link_forge_inputs.sh hard-links it), so the rivers land on the model's own cells.
GRID_PATH = "/anvil/scratch/x-uheede/cstar/_forge_bp_runs/Iceland2_NEW_customriver_Iceland2_NEW_256procs/input_data/Iceland2_NEW_customriver_Iceland2_NEW_256procs_grid.nc"
GLOFAS_PATH = "/anvil/projects/x-ees250129/Datasets/Rivers/processed_GloFAS/glofas_v4_rivers_daily.nc"
HVAL_PATH = "/anvil/projects/x-ees250129/Datasets/Iceland_river_dataset/Hvalfjordur_rivers_2024.nc"

# The river custom_file of Iceland2_glodap_woa.forge_blueprint.yaml
OUT_PATH = "/anvil/scratch/x-uheede/iceland2_glodap_woa/Iceland2_rivers_glofas_hval.nc"

# The file the iceland2_customriver runs used; summarized at the end for comparison
OLD_FILE = "/anvil/projects/x-ees250129/x-uheede/INPUT_files/Iceland2_MARBL_2024/RT_EMOD/Iceland2_river_combied.nc"

START, END = datetime(2023, 1, 1), datetime(2024, 12, 31)
MODEL_REFERENCE_DATE = datetime(2000, 1, 1)

In [ ]:
# Hvalfjörður: GloFAS rivers with a cell in this box are dropped
FJORD_LAT = (64.25, 64.45)
FJORD_LON = (338.1, 338.8)  # degrees east: 21.9 W to 21.2 W

# Hvalfjörður discharge: every month is half the August flow, and 40 % of that in these months
WINTER_MONTHS = (1, 2, 3, 4, 11, 12)

# River temperature (all rivers): 0 to T_MAX over the year
T_MAX = 10.0

# Table S1: Hvalfjörður river concentrations (mmol m-3; salinity in psu).
# Every other tracer of the Hvalfjörður rivers is 0, Fe and Lig included.
TABLE_S1 = {
    "salt": 1.0,
    "ALK": 282.0, "ALK_ALT_CO2": 282.0,
    "DIC": 313.0, "DIC_ALT_CO2": 313.0,
    "NO3": 12.0, "PO4": 0.9, "SiO3": 14.0, "NH4": 0.7, "O2": 300.0,
}

In [ ]:
def names(da):
    # Strings from a str, bytes or char-array variable
    v = da.values
    if v.dtype.kind == "S" and v.ndim == 2:
        return [b"".join(r).decode().strip() for r in v]
    return [x.decode().strip() if isinstance(x, bytes) else str(x).strip() for x in v.ravel()]


def monthly_dates(years=(2023, 2024)):
    return pd.DatetimeIndex([datetime(y, m, 15) for y in years for m in range(1, 13)])


def to_days(times):
    # Days since MODEL_REFERENCE_DATE, float64
    return ((pd.DatetimeIndex(times) - pd.Timestamp(MODEL_REFERENCE_DATE)) / pd.Timedelta(days=1)).to_numpy(float)


def monthly_to_daily(values_by_month, daily_times):
    # Interpolate a (24, ...) array on the 15th of each month of 2023-2024 to daily_times;
    # values before the first or after the last 15th stay constant
    x, xi = to_days(monthly_dates()), to_days(daily_times)
    flat = values_by_month.reshape(len(x), -1)
    out = np.stack([np.interp(xi, x, flat[:, j]) for j in range(flat.shape[1])], axis=1)
    return out.reshape((len(xi),) + values_by_month.shape[1:])


def river_cells(ds):
    # {river number: [(eta, xi), ...]} from the river_index map
    idx = ds["river_index"].values
    cells = {}
    for eta, xi in zip(*np.nonzero(idx > 0)):
        cells.setdefault(int(round(idx[eta, xi])), []).append((int(eta), int(xi)))
    return cells


def tracer_ranges(ds, rivers=None):
    # min .. max of every non-zero tracer, over time and the given rivers (indices along nriver)
    tr = ds["river_tracer"]
    tdim = next(d for d in tr.dims if d.startswith("ntrac"))
    if rivers is not None:
        tr = tr.isel(nriver=rivers)
    rows = []
    for i, name in enumerate(names(ds["tracer_name"])):
        a = tr.isel({tdim: i})
        lo, hi = float(a.min()), float(a.max())
        if lo != 0 or hi != 0:
            rows.append((name, lo, hi))
    return pd.DataFrame(rows, columns=["tracer", "min", "max"]).set_index("tracer")


def hash_netcdf_contents(path):
    # The content hash forge records for a user-provided file: a copy of
    # cstar.applications.forge.user_files.hash_netcdf_contents (cstar-ocean 0.16.1)
    def _attr_bytes(attrs):
        return b"|".join(f"{k}={np.asarray(attrs[k]).tolist()!r}".encode() for k in sorted(attrs))

    def _value_bytes(var):
        values = var.values
        if values.dtype.kind in "OUS":
            flat = np.asarray(values).ravel()
            return b"\x00".join(x if isinstance(x, bytes) else str(x).encode() for x in flat.tolist())
        return np.ascontiguousarray(values).tobytes()

    with xr.open_dataset(path, decode_cf=False, decode_times=False) as ds:
        ds.load()
        h = hashlib.sha256()
        h.update(_attr_bytes(ds.attrs))
        for name in sorted(ds.variables):
            var = ds.variables[name]
            h.update(b"||VAR||")
            h.update(name.encode())
            h.update(repr(tuple(var.dims)).encode())
            h.update(repr(tuple(var.shape)).encode())
            h.update(str(var.dtype).encode())
            h.update(_value_bytes(var))
            h.update(_attr_bytes(var.attrs))
    return h.hexdigest()

## Grid

In [ ]:
grid = Grid._from_file(GRID_PATH)
lat = grid.ds["lat_rho"].values
lon = grid.ds["lon_rho"].values % 360
in_fjord = (lat >= FJORD_LAT[0]) & (lat <= FJORD_LAT[1]) & (lon >= FJORD_LON[0]) & (lon <= FJORD_LON[1])
print(f"grid {lat.shape}, {int(in_fjord.sum())} cells in the Hvalfjörður box")

## GloFAS rivers

roms-tools places each GloFAS river on the nearest coastal cell and numbers the rivers largest first. Rivers that share a cell are merged into one `overlap_<name>` river, named after the alphabetically first of them, which carries their combined discharge; the merged rivers keep zero discharge and no cell. roms-tools also sets the discharge of every GloFAS river with a mean under 1 m³/s to zero (its `min_discharge_m3s` default); such a river keeps its number and its cell. Pass `min_discharge_m3s=0` to keep the small rivers. With `bgc_source` `CONSTANTS`, every tracer gets the roms-tools default concentration; the temperature is replaced further down.

In [ ]:
glofas = RiverForcing(
    grid=grid,
    start_time=START,
    end_time=END,
    source={"name": "GLOFAS", "path": GLOFAS_PATH},
    convert_to_climatology="never",
    include_bgc=True,
    bgc_source={"name": "CONSTANTS"},
    model_reference_date=MODEL_REFERENCE_DATE,
).ds.load()
print(f"{glofas.sizes['nriver']} GloFAS rivers, {glofas.sizes['river_time']} daily records, "
      f"{pd.Timestamp(glofas.abs_time.values[0]):%Y-%m-%d} to {pd.Timestamp(glofas.abs_time.values[-1]):%Y-%m-%d}")

## Hvalfjörður rivers

The custom dataset has the Dai & Trenberth layout: monthly discharge for 2024. Rivers that share a cell are merged as above. On the Iceland2 grid that gives `overlap_HVAL7` (HVAL7 + HVAL8) and `overlap_HVAL12` (HVAL12 + HVAL13), so the table below lists HVAL7, HVAL8, HVAL12 and HVAL13 with zero flow: they have no cell, and their flow is in the overlap rivers, which count as Hvalfjörður rivers. The 1 m³/s floor does not apply to this source.

In [ ]:
hval = RiverForcing(
    grid=grid,
    start_time=datetime(2024, 1, 1),
    end_time=datetime(2024, 12, 31),
    source={"name": "DAI", "path": HVAL_PATH, "climatology": False},
    convert_to_climatology="never",
    include_bgc=True,
    bgc_source={"name": "CONSTANTS"},
    model_reference_date=MODEL_REFERENCE_DATE,
).ds.load()
h_names = names(hval["river_name"])
print(f"{len(h_names)} Hvalfjörður rivers: {', '.join(h_names)}")

tracer_names = names(glofas["tracer_name"])
assert names(hval["tracer_name"]) == tracer_names, "the two sets list different tracers"

### Discharge

As in the earlier version: every month gets half the August discharge, and 40 % of that in January–April and November–December. 2023 repeats 2024. The values sit on the 15th of each month.

In [ ]:
h_months = list(pd.DatetimeIndex(hval["abs_time"].values).month)
assert sorted(h_months) == list(range(1, 13)), f"expected 12 monthly records for 2024, got months {h_months}"
august = hval["river_volume"].transpose("river_time", "nriver").values[h_months.index(8)]
factor = np.array([0.4 if m in WINTER_MONTHS else 1.0 for m in range(1, 13)])
h_vol_month = np.tile(0.5 * factor[:, None] * august[None, :], (2, 1))  # Jan 2023 .. Dec 2024

pd.DataFrame({"August flow (m3/s)": august, "summer (m3/s)": 0.5 * august, "winter (m3/s)": 0.2 * august},
             index=h_names).round(2)

## GloFAS rivers inside Hvalfjörður

A GloFAS river with any cell in the Hvalfjörður box is dropped, discharge included. No cell may hold both a GloFAS river and a Hvalfjörður river. A dropped river with zero mean discharge is one under the 1 m³/s floor.

In [ ]:
g_names = names(glofas["river_name"])
g_cells, h_cells = river_cells(glofas), river_cells(hval)
g_volume = glofas["river_volume"].transpose("river_time", "nriver").values

drop = sorted(k for k, cells in g_cells.items() if any(in_fjord[c] for c in cells))
keep = [k for k in range(1, glofas.sizes["nriver"] + 1) if k not in drop]
display(pd.DataFrame({"river": [g_names[k - 1] for k in drop],
                      "mean discharge (m3/s)": [g_volume[:, k - 1].mean() for k in drop]},
                     index=pd.Index(drop, name="GloFAS number")).round(2))

g_map, h_map = glofas["river_index"].values, hval["river_index"].values
shared = np.argwhere((g_map > 0) & (h_map > 0) & ~np.isin(np.round(g_map), drop))
assert len(shared) == 0, f"{len(shared)} cells hold both a GloFAS and a Hvalfjörður river, e.g. {shared[0]}"
n_merged = sum(1 for k in keep if k not in g_cells)
n_zero = sum(1 for k in keep if k in g_cells and not g_volume[:, k - 1].any())
print(f"{len(keep)} GloFAS rivers kept, {len(drop)} dropped. Of the kept rivers, {n_merged} have no cell "
      f"(merged into overlap_ rivers) and {n_zero} have zero discharge (under the 1 m3/s floor)")

## Combined forcing

Everything goes on GloFAS's daily time axis. The Hvalfjörður values and the temperature are interpolated linearly between the 15th of each month, as ROMS does with monthly records. Rivers are numbered once: the kept GloFAS rivers first, then the Hvalfjörður rivers.

In [ ]:
times = pd.DatetimeIndex(glofas["abs_time"].values)
ng, nh = len(keep), len(h_names)

volume = np.concatenate([g_volume[:, [k - 1 for k in keep]], monthly_to_daily(h_vol_month, times)], axis=1)

# tracers: GloFAS rivers keep the roms-tools defaults, Hvalfjörður rivers get Table S1
tracer = np.zeros((len(times), len(tracer_names), ng + nh))
g_tracer = glofas["river_tracer"].transpose("river_time", "ntracers", "nriver").values
tracer[:, :, :ng] = g_tracer[:, :, [k - 1 for k in keep]]
for i, name in enumerate(tracer_names):
    tracer[:, i, ng:] = TABLE_S1.get(name, 0.0)

# temperature, all rivers
temp_month = np.tile(T_MAX * np.sin(np.linspace(0, np.pi, 12)), 2)
tracer[:, tracer_names.index("temp"), :] = monthly_to_daily(temp_month, times)[:, None]

# river location maps with the new numbering
new_number = {k: j + 1 for j, k in enumerate(keep)}
index = np.zeros(g_map.shape)
fraction = np.zeros(g_map.shape)
g_frac, h_frac = glofas["river_fraction"].values, hval["river_fraction"].values
for k, cells in g_cells.items():
    if k in new_number:
        for c in cells:
            index[c], fraction[c] = new_number[k], g_frac[c]
for k, cells in h_cells.items():
    for c in cells:
        index[c], fraction[c] = ng + k, h_frac[c]

placed = set(np.unique(index[index > 0]).astype(int))
print(f"{ng} GloFAS + {nh} Hvalfjörður rivers, {len(times)} daily records; "
      f"{ng + nh - len(placed)} rivers without a cell")

In [ ]:
out = xr.Dataset(
    {
        "river_volume": (("river_time", "nriver"), volume, glofas["river_volume"].attrs),
        "river_tracer": (("river_time", "ntracers", "nriver"), tracer, glofas["river_tracer"].attrs),
        "river_index": (glofas["river_index"].dims, index, glofas["river_index"].attrs),
        "river_fraction": (glofas["river_fraction"].dims, fraction, glofas["river_fraction"].attrs),
    },
    coords={
        "river_time": ("river_time", to_days(times), {
            "long_name": f"relative time: days since {MODEL_REFERENCE_DATE:%Y-%m-%d %H:%M:%S}",
            "units": "days"}),
        "abs_time": ("river_time", to_days(times), {
            "long_name": "absolute time",
            "units": f"days since {MODEL_REFERENCE_DATE:%Y-%m-%d %H:%M:%S}",
            "calendar": "proleptic_gregorian"}),
        "nriver": ("nriver", np.arange(1, ng + nh + 1, dtype=np.int32),
                   {"long_name": "River ID (1-based Fortran indexing)"}),
        "river_name": ("nriver", [g_names[k - 1] for k in keep] + h_names, {"long_name": "River name"}),
        "tracer_name": ("ntracers", tracer_names, glofas["tracer_name"].attrs),
    },
    attrs={
        "title": "Iceland2 river forcing: GloFAS rivers and the Hvalfjordur rivers",
        "source": f"Iceland2_river_forcing.ipynb; GloFAS: {Path(GLOFAS_PATH).name}; Hvalfjordur: {Path(HVAL_PATH).name}",
        "tracers": "temperature 0-10 C (sinusoid over the year) for all rivers; Hvalfjordur rivers: Table S1, "
                   "other tracers 0; GloFAS rivers: roms-tools default concentrations",
        "hvalfjordur_discharge": "0.5 x August flow every month, x 0.4 in Jan-Apr and Nov-Dec; 2023 repeats 2024",
        "created": datetime.now().strftime("%Y-%m-%d %H:%M"),
    },
)
for extra in ("tracer_unit", "tracer_long_name"):
    if extra in glofas:
        out = out.assign_coords({extra: ("ntracers", names(glofas[extra]), glofas[extra].attrs)})
out

## Check

In [ ]:
lon_deg = np.where(lon > 180, lon - 360, lon)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, extent in zip(axes, [None, (-22.3, -21.0, 64.2, 64.55)]):
    ax.pcolormesh(lon_deg, lat, grid.ds["mask_rho"].values, cmap="Greys_r", vmin=-1, vmax=1, shading="auto")
    for k in keep:
        for eta, xi in g_cells.get(k, []):
            ax.plot(lon_deg[eta, xi], lat[eta, xi], "o", color="tab:blue", ms=4)
    for k in drop:
        for eta, xi in g_cells[k]:
            ax.plot(lon_deg[eta, xi], lat[eta, xi], "x", color="tab:gray", ms=7)
    for cells in h_cells.values():
        for eta, xi in cells:
            ax.plot(lon_deg[eta, xi], lat[eta, xi], "*", color="tab:red", ms=9)
    ax.add_patch(plt.Rectangle((FJORD_LON[0] - 360, FJORD_LAT[0]), FJORD_LON[1] - FJORD_LON[0],
                               FJORD_LAT[1] - FJORD_LAT[0], fill=False, ls="--", color="tab:orange"))
    if extent:
        ax.set_xlim(extent[:2])
        ax.set_ylim(extent[2:])
    ax.set_xlabel("longitude")
    ax.set_ylabel("latitude")
axes[0].set_title("River cells: GloFAS kept (blue), GloFAS dropped (grey), Hvalfjörður (red)")
axes[1].set_title("Hvalfjörður; dashed: the box that drops GloFAS rivers");

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax1.plot(times, volume[:, :ng].sum(axis=1), label="GloFAS rivers")
ax1.plot(times, volume[:, ng:].sum(axis=1), label="Hvalfjörður rivers")
ax1.set_ylabel("total discharge (m$^3$ s$^{-1}$)")
ax1.legend()
ax2.plot(times, tracer[:, tracer_names.index("temp"), 0], color="tab:red")
ax2.set_ylabel("river temperature (°C)");

## Save

CDF5 (`NETCDF3_64BIT_DATA`), the format ParallelIO reads, with every floating-point variable, the times included, in double precision. The content hash goes into the river `custom_file` of `Iceland2_glodap_woa.forge_blueprint.yaml`; forge only warns if it differs.

In [ ]:
Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)
assert not Path(OUT_PATH).exists(), f"{OUT_PATH} exists; remove it first"
encoding = {v: {"dtype": "float64", "_FillValue": None} for v in
            ("river_volume", "river_tracer", "river_index", "river_fraction", "river_time", "abs_time")}
out.to_netcdf(OUT_PATH, format="NETCDF3_64BIT_DATA", encoding=encoding)

with xr.open_dataset(OUT_PATH, decode_times=False, decode_timedelta=False) as check:
    int64 = [n for n, v in check.variables.items() if v.dtype == "int64"]
    assert not int64, f"int64 variables: {int64}"
    print(f"wrote {OUT_PATH}")
    print(f"  dims {dict(check.sizes)}; river_index dims {check['river_index'].dims}")
    print(f"  dtypes {sorted({str(v.dtype) for v in check.variables.values()})}")
print(f"content_hash: {hash_netcdf_contents(OUT_PATH)}")

In [ ]:
with xr.open_dataset(OUT_PATH, decode_times=False, decode_timedelta=False) as saved:
    display(pd.concat({"GloFAS rivers": tracer_ranges(saved, slice(0, ng)),
                       "Hvalfjörður rivers": tracer_ranges(saved, slice(ng, ng + nh))}, axis=1).fillna(0))

## The earlier file

The file the `iceland2_customriver` runs used, for comparison: its rivers and its non-zero tracer values.

In [ ]:
if Path(OLD_FILE).exists():
    with xr.open_dataset(OLD_FILE, decode_times=False, decode_timedelta=False) as old:
        print(f"{old.sizes['nriver']} rivers: {', '.join(names(old['river_name']))}")
        print(f"river_index dims {old['river_index'].dims}; time dtypes "
              f"{[str(old[v].dtype) for v in ('river_time', 'abs_time') if v in old]}")
        display(tracer_ranges(old))
else:
    print(f"{OLD_FILE} not found")